# Subreddit Analysis Across Train/Val/Test Splits

This notebook analyzes:
1. Number of unique subreddits in each split
2. Number of unique subreddits in val/test that don't appear in train
3. Percentage of instances in val/test that come from unseen subreddits

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import yaml
from ssf.Constants import *
from ssf.Configs import load_config

config = load_config(REPLICATION_CONFIG_PATH)

## Load Data

In [ ]:
# Load splits
train_df = pd.read_csv(f"{config.dirs.data.corpus}/ssf_split_train.csv")
val_df = pd.read_csv(f"{config.dirs.data.corpus}/ssf_split_val.csv")
test_df = pd.read_csv(f"{config.dirs.data.corpus}/ssf_split_test.csv")

print(f"Train size: {len(train_df):,}")
print(f"Val size: {len(val_df):,}")
print(f"Test size: {len(test_df):,}")

## Extract Subreddit Sets

In [ ]:
# Get unique subreddits in each split
train_subreddits = set(train_df['meta.subreddit'].dropna())
val_subreddits = set(val_df['meta.subreddit'].dropna())
test_subreddits = set(test_df['meta.subreddit'].dropna())

print(f"Unique subreddits in train: {len(train_subreddits):,}")
print(f"Unique subreddits in val: {len(val_subreddits):,}")
print(f"Unique subreddits in test: {len(test_subreddits):,}")

## Calculate Unseen Subreddits

In [ ]:
# Find subreddits in val/test that are NOT in train
unseen_val_subreddits = val_subreddits - train_subreddits
unseen_test_subreddits = test_subreddits - train_subreddits

print("\n" + "="*60)
print("UNIQUE SUBREDDITS NOT IN TRAIN")
print("="*60)
print(f"Val subreddits not in train: {len(unseen_val_subreddits):,}")
print(f"Test subreddits not in train: {len(unseen_test_subreddits):,}")

## Calculate Percentage of Instances from Unseen Subreddits

In [ ]:
# Count instances from unseen subreddits
val_unseen_instances = val_df[val_df['meta.subreddit'].isin(unseen_val_subreddits)]
test_unseen_instances = test_df[test_df['meta.subreddit'].isin(unseen_test_subreddits)]

val_unseen_pct = (len(val_unseen_instances) / len(val_df)) * 100
test_unseen_pct = (len(test_unseen_instances) / len(test_df)) * 100

print("\n" + "="*60)
print("PERCENTAGE OF INSTANCES FROM UNSEEN SUBREDDITS")
print("="*60)
print(f"Val instances from unseen subreddits: {len(val_unseen_instances):,} / {len(val_df):,} ({val_unseen_pct:.2f}%)")
print(f"Test instances from unseen subreddits: {len(test_unseen_instances):,} / {len(test_df):,} ({test_unseen_pct:.2f}%)")

## Summary Statistics

In [ ]:
# Create summary dataframe
summary = pd.DataFrame({
    'Split': ['Train', 'Val', 'Test'],
    'Total Instances': [len(train_df), len(val_df), len(test_df)],
    'Unique Subreddits': [len(train_subreddits), len(val_subreddits), len(test_subreddits)],
    'Unseen Subreddits': [0, len(unseen_val_subreddits), len(unseen_test_subreddits)],
    'Unseen Instances': [0, len(val_unseen_instances), len(test_unseen_instances)],
    'Unseen %': [0.0, val_unseen_pct, test_unseen_pct]
})

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(summary.to_string(index=False))

## Optional: View Sample Unseen Subreddits

In [ ]:
# Show some examples of unseen subreddits
print("\nSample unseen subreddits in Val (first 10):")
print(list(unseen_val_subreddits)[:10])

print("\nSample unseen subreddits in Test (first 10):")
print(list(unseen_test_subreddits)[:10])

## Optional: Distribution of Instances per Subreddit

In [ ]:
# Check if unseen subreddits tend to have fewer instances
val_subreddit_counts = val_df['meta.subreddit'].value_counts()
test_subreddit_counts = test_df['meta.subreddit'].value_counts()

print("\nVal - Instances per unseen subreddit statistics:")
unseen_val_counts = val_subreddit_counts[val_subreddit_counts.index.isin(unseen_val_subreddits)]
print(unseen_val_counts.describe())

print("\nTest - Instances per unseen subreddit statistics:")
unseen_test_counts = test_subreddit_counts[test_subreddit_counts.index.isin(unseen_test_subreddits)]
print(unseen_test_counts.describe())